# EMT Ph3 Generic Four-Terminal V-Type SSN — 6×6 Interface Test vs MNA

This notebook tests the new generic four-terminal V-type SSN implementation with the convention:

```text
v0   v1
v2   v3

vin  = v2 - v0
vout = v3 - v1
```

The SSN interface vectors are therefore:

```text
u = [vin_abc;  vout_abc]   # 6×1
y = [iin_abc;  iout_abc]   # 6×1
```

The test compares:

1. a native MNA reference using two independent three-phase series RL branches, and
2. one `GenericFourTerminalVTypeSSN` containing the two branches as one 6-input/6-output state-space model.

No terminal of the SSN block is internally grounded. Grounding only comes from the surrounding source/load network.

## Test topology

The SSN block has four three-phase terminals:

```text
terminal 0 = nL_ret       terminal 1 = nR_ret
terminal 2 = nL_send      terminal 3 = nR_send
```

Thus:

```text
vin  = v(nL_send) - v(nL_ret)
vout = v(nR_send) - v(nR_ret)
```

The physical test network is:

```text
left source  -> nL_send -> SSN branch in  -> nL_ret -> left load  -> ground
right source -> nR_send -> SSN branch out -> nR_ret -> right load -> ground
```

The native MNA reference replaces the SSN block by two explicit `Resistor + Inductor` branches.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import dpsimpy as dpsim

## Parameters

In [ ]:
time_step = 0.00005
final_time = 0.1
frequency = 50.0

# Use two different source phasors so both SSN ports are exercised independently.
left_source_voltage = 1.0 * np.exp(1j * 0.0)
right_source_voltage = 0.8 * np.exp(-1j * np.deg2rad(20.0))

branch_resistance = dpsim.Math.single_phase_parameter_to_three_phase(0.2)
branch_inductance = dpsim.Math.single_phase_parameter_to_three_phase(0.02)

left_load_resistance = dpsim.Math.single_phase_parameter_to_three_phase(10.0)
right_load_resistance = dpsim.Math.single_phase_parameter_to_three_phase(12.0)

## State-space matrices for the four-terminal SSN

State vector:

```text
x = [iin_abc; iout_abc]      # 6×1
```

Input vector:

```text
u = [vin_abc; vout_abc]      # 6×1
```

Branch equations:

```text
L d(iin)/dt  = vin  - R iin
L d(iout)/dt = vout - R iout
```

Output:

```text
y = [iin_abc; iout_abc]
```

So the matrix dimensions are:

```text
A: 6×6
B: 6×6
C: 6×6
D: 6×6
```

In [ ]:
def create_four_terminal_two_port_rl_matrices():
    inv_l = np.linalg.inv(branch_inductance)
    i3 = np.eye(3)

    a = np.zeros((6, 6))
    b = np.zeros((6, 6))
    c = np.zeros((6, 6))
    d = np.zeros((6, 6))

    # Input branch: L d(iin)/dt = vin - R iin
    a[0:3, 0:3] = -inv_l @ branch_resistance
    b[0:3, 0:3] = inv_l

    # Output branch: L d(iout)/dt = vout - R iout
    a[3:6, 3:6] = -inv_l @ branch_resistance
    b[3:6, 3:6] = inv_l

    # y = [iin; iout]
    c[0:3, 0:3] = i3
    c[3:6, 3:6] = i3

    return a, b, c, d

## Native MNA reference simulation

This model uses explicit native components:

```text
nL_send -> R -> L -> nL_ret
nR_send -> R -> L -> nR_ret
```

In [ ]:
def run_reference_mna():
    sim_name = "PY_EMT_Ph3_FourTerminal_MNA_Reference"

    n_l_send = dpsim.emt.SimNode("nL_send", dpsim.PhaseType.ABC)
    n_l_mid = dpsim.emt.SimNode("nL_mid", dpsim.PhaseType.ABC)
    n_l_ret = dpsim.emt.SimNode("nL_ret", dpsim.PhaseType.ABC)

    n_r_send = dpsim.emt.SimNode("nR_send", dpsim.PhaseType.ABC)
    n_r_mid = dpsim.emt.SimNode("nR_mid", dpsim.PhaseType.ABC)
    n_r_ret = dpsim.emt.SimNode("nR_ret", dpsim.PhaseType.ABC)

    vs_left = dpsim.emt.ph3.VoltageSource("VS_Left")
    vs_left.set_parameters(
        dpsim.Math.single_phase_variable_to_three_phase(left_source_voltage),
        frequency,
    )

    vs_right = dpsim.emt.ph3.VoltageSource("VS_Right")
    vs_right.set_parameters(
        dpsim.Math.single_phase_variable_to_three_phase(right_source_voltage),
        frequency,
    )

    r_left = dpsim.emt.ph3.Resistor("R_Left")
    r_left.set_parameters(branch_resistance)

    l_left = dpsim.emt.ph3.Inductor("L_Left")
    l_left.set_parameters(branch_inductance)

    r_right = dpsim.emt.ph3.Resistor("R_Right")
    r_right.set_parameters(branch_resistance)

    l_right = dpsim.emt.ph3.Inductor("L_Right")
    l_right.set_parameters(branch_inductance)

    load_left = dpsim.emt.ph3.Resistor("Load_Left")
    load_left.set_parameters(left_load_resistance)

    load_right = dpsim.emt.ph3.Resistor("Load_Right")
    load_right.set_parameters(right_load_resistance)

    vs_left.connect([dpsim.emt.SimNode.gnd, n_l_send])
    r_left.connect([n_l_send, n_l_mid])
    l_left.connect([n_l_mid, n_l_ret])
    load_left.connect([n_l_ret, dpsim.emt.SimNode.gnd])

    vs_right.connect([dpsim.emt.SimNode.gnd, n_r_send])
    r_right.connect([n_r_send, n_r_mid])
    l_right.connect([n_r_mid, n_r_ret])
    load_right.connect([n_r_ret, dpsim.emt.SimNode.gnd])

    system = dpsim.SystemTopology(
        frequency,
        [n_l_send, n_l_mid, n_l_ret, n_r_send, n_r_mid, n_r_ret],
        [
            vs_left,
            vs_right,
            r_left,
            l_left,
            r_right,
            l_right,
            load_left,
            load_right,
        ],
    )

    dpsim.Logger.set_log_dir("logs/" + sim_name)
    logger = dpsim.Logger(sim_name)
    logger.log_attribute("vL_send", n_l_send.attr("v"))
    logger.log_attribute("vL_ret", n_l_ret.attr("v"))
    logger.log_attribute("vR_send", n_r_send.attr("v"))
    logger.log_attribute("vR_ret", n_r_ret.attr("v"))
    logger.log_attribute("iL_branch", l_left.attr("i_intf"))
    logger.log_attribute("iR_branch", l_right.attr("i_intf"))
    logger.log_attribute("iLoadLeft", load_left.attr("i_intf"))
    logger.log_attribute("iLoadRight", load_right.attr("i_intf"))

    sim = dpsim.Simulation(sim_name)
    sim.set_system(system)
    sim.add_logger(logger)
    sim.set_domain(dpsim.Domain.EMT)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.run()

    return sim_name

## Four-terminal SSN simulation

The SSN connection is the important part:

```python
ssn.connect([n_l_ret, n_r_ret, n_l_send, n_r_send])
```

because the four terminals are interpreted as:

```text
v0 = n_l_ret
v1 = n_r_ret
v2 = n_l_send
v3 = n_r_send
```

Therefore:

```text
vin  = v2 - v0 = v(n_l_send) - v(n_l_ret)
vout = v3 - v1 = v(n_r_send) - v(n_r_ret)
```

In [ ]:
def run_four_terminal_ssn():
    sim_name = "PY_EMT_Ph3_GenericFourTerminalVTypeSSN_TwoPort_RL"

    n_l_send = dpsim.emt.SimNode("nL_send", dpsim.PhaseType.ABC)
    n_l_ret = dpsim.emt.SimNode("nL_ret", dpsim.PhaseType.ABC)

    n_r_send = dpsim.emt.SimNode("nR_send", dpsim.PhaseType.ABC)
    n_r_ret = dpsim.emt.SimNode("nR_ret", dpsim.PhaseType.ABC)

    a, b, c, d = create_four_terminal_two_port_rl_matrices()

    vs_left = dpsim.emt.ph3.VoltageSource("VS_Left")
    vs_left.set_parameters(
        dpsim.Math.single_phase_variable_to_three_phase(left_source_voltage),
        frequency,
    )

    vs_right = dpsim.emt.ph3.VoltageSource("VS_Right")
    vs_right.set_parameters(
        dpsim.Math.single_phase_variable_to_three_phase(right_source_voltage),
        frequency,
    )

    ssn = dpsim.emt.ph3.GenericFourTerminalVTypeSSN("FourTerminalSSN")
    ssn.set_parameters(a, b, c, d)

    load_left = dpsim.emt.ph3.Resistor("Load_Left")
    load_left.set_parameters(left_load_resistance)

    load_right = dpsim.emt.ph3.Resistor("Load_Right")
    load_right.set_parameters(right_load_resistance)

    vs_left.connect([dpsim.emt.SimNode.gnd, n_l_send])
    vs_right.connect([dpsim.emt.SimNode.gnd, n_r_send])

    # Four-terminal convention:
    #
    #   v0   v1
    #   v2   v3
    #
    #   vin  = v2 - v0
    #   vout = v3 - v1
    #
    # So connect:
    #   terminal 0 -> left return
    #   terminal 1 -> right return
    #   terminal 2 -> left send
    #   terminal 3 -> right send
    ssn.connect([n_l_ret, n_r_ret, n_l_send, n_r_send])

    load_left.connect([n_l_ret, dpsim.emt.SimNode.gnd])
    load_right.connect([n_r_ret, dpsim.emt.SimNode.gnd])

    system = dpsim.SystemTopology(
        frequency,
        [n_l_send, n_l_ret, n_r_send, n_r_ret],
        [vs_left, vs_right, ssn, load_left, load_right],
    )

    dpsim.Logger.set_log_dir("logs/" + sim_name)
    logger = dpsim.Logger(sim_name)
    logger.log_attribute("vL_send", n_l_send.attr("v"))
    logger.log_attribute("vL_ret", n_l_ret.attr("v"))
    logger.log_attribute("vR_send", n_r_send.attr("v"))
    logger.log_attribute("vR_ret", n_r_ret.attr("v"))
    logger.log_attribute("iSSN", ssn.attr("i_intf"))
    logger.log_attribute("vSSN", ssn.attr("v_intf"))
    logger.log_attribute("iLoadLeft", load_left.attr("i_intf"))
    logger.log_attribute("iLoadRight", load_right.attr("i_intf"))

    sim = dpsim.Simulation(sim_name)
    sim.set_system(system)
    sim.add_logger(logger)
    sim.set_domain(dpsim.Domain.EMT)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.run()

    return sim_name

## Run simulations

In [ ]:
mna_sim_name = run_reference_mna()
ssn_sim_name = run_four_terminal_ssn()

print("MNA reference log:", Path("logs") / mna_sim_name / f"{mna_sim_name}.csv")
print("SSN log:", Path("logs") / ssn_sim_name / f"{ssn_sim_name}.csv")

## Load logs

In [ ]:
def load_log(sim_name):
    path = Path("logs") / sim_name / f"{sim_name}.csv"
    df = pd.read_csv(path, skipinitialspace=True)
    df.columns = df.columns.str.strip()
    return df


mna = load_log(mna_sim_name)
ssn = load_log(ssn_sim_name)

display(mna.head())
display(ssn.head())

print("MNA columns:")
print(list(mna.columns))
print("\nSSN columns:")
print(list(ssn.columns))

## Helper functions

In [ ]:
def first_matching_column(df, prefix):
    matches = [c for c in df.columns if c.startswith(prefix)]
    if not matches:
        matches = [c for c in df.columns if prefix in c]
    if not matches:
        raise KeyError(
            f"No column matching {prefix!r}. Available columns: {list(df.columns)}"
        )
    return matches[0]


def all_matching_columns(df, prefix):
    matches = [c for c in df.columns if c.startswith(prefix)]
    if not matches:
        matches = [c for c in df.columns if prefix in c]
    return matches


def time_column(df):
    for candidate in ["time", "Time", "t"]:
        if candidate in df.columns:
            return candidate
    return df.columns[0]


def compare_column(prefix_mna, prefix_ssn, title, ylabel):
    t_mna = time_column(mna)
    t_ssn = time_column(ssn)

    col_mna = first_matching_column(mna, prefix_mna)
    col_ssn = first_matching_column(ssn, prefix_ssn)

    plt.figure()
    plt.plot(mna[t_mna], mna[col_mna], label=f"MNA {col_mna}")
    plt.plot(ssn[t_ssn], ssn[col_ssn], "--", label=f"SSN {col_ssn}")
    plt.xlabel("Time [s]")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

    n = min(len(mna), len(ssn))
    err = np.asarray(ssn[col_ssn].iloc[:n]) - np.asarray(mna[col_mna].iloc[:n])
    print(f"max abs error {title}: {np.max(np.abs(err))}")

## Compare left return voltage

In [ ]:
compare_column("vL_ret", "vL_ret", "Left return voltage", "Voltage")

## Compare right return voltage

In [ ]:
compare_column("vR_ret", "vR_ret", "Right return voltage", "Voltage")

## Compare left load current

In [ ]:
compare_column("iLoadLeft", "iLoadLeft", "Left load current", "Current")

## Compare right load current

In [ ]:
compare_column("iLoadRight", "iLoadRight", "Right load current", "Current")

## Check SSN interface voltage columns

For the SSN implementation, `vSSN` should contain six components:

```text
[v2_a-v0_a, v2_b-v0_b, v2_c-v0_c,
 v3_a-v1_a, v3_b-v1_b, v3_c-v1_c]
```

This cell lists the relevant columns so you can verify that the logger exports the expected 6 interface voltages and 6 interface currents.

In [ ]:
print("vSSN columns:")
print(all_matching_columns(ssn, "vSSN"))

print("\niSSN columns:")
print(all_matching_columns(ssn, "iSSN"))

## Interpretation

Expected result:

- `vL_ret` should match between MNA and SSN.
- `vR_ret` should match between MNA and SSN.
- `iLoadLeft` and `iLoadRight` should match.
- `vSSN` should have 6 components, not 12.
- `iSSN` should have 6 interface currents, not terminal currents.

The four-terminal terminal-current mapping is handled inside `FourTerminalVTypeSSNComp` by:

```text
i_terminal = T.T @ i_interface
```

with:

```text
T = [ -I   0   I   0
       0  -I   0   I ]
```

So the physical terminal injections are:

```text
terminal 0: -iin
terminal 2: +iin
terminal 1: -iout
terminal 3: +iout
```